# Guía rápida para completar la tabla ML

Ejecuta las secciones **1 a 6**. Al terminar la sección 6 tendrás los dos CSV de estabilidad por procedencia para los modelos ML. La sección **7** es opcional.


# 1. Imports y configuración - EJECUTAR

Define rutas, datasets, `OUTDIR`, métricas y librerías.


In [1]:
# ==============================================================================
# 1. IMPORTAR LIBRERÍAS Y CONFIGURACIÓN
# ==============================================================================
import os, re, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score

# Scikit-learn
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, 
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

# Weights & Biases
import wandb

# Joblib para guardar modelos
import joblib

# ========== CONFIGURACIÓN ==========
DATA_ROOT   = Path(r"C:\Users\Victoria\Desktop\data\datos_raw")
DATASETS    = ["aptos_2019", "eyePACS", "idrid", "messidor"]
N_CLASSES   = 5

# Parámetros de datos
SEED        = 42
VAL_SIZE    = 0.15
BALANCE_MAX_PER_CLASS = 1000

# Ruta de salida
BASE_OUT    = Path(r"C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml")
BASE_OUT.mkdir(parents=True, exist_ok=True)
OUTDIR      = BASE_OUT / "COMBINED_TRAIN_ONLY"
OUTDIR.mkdir(parents=True, exist_ok=True)

# W&B
WANDB_LOCAL_DIR = Path(r"C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\_wandb_runs")
WANDB_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_DIR"] = str(WANDB_LOCAL_DIR)

# Seed
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
set_seed(SEED)

print("[CONFIG] Rutas y configuración inicializadas.")
print(f"  OUTDIR: {OUTDIR}")
print(f"  WANDB_LOCAL_DIR: {WANDB_LOCAL_DIR}")


[CONFIG] Rutas y configuración inicializadas.
  OUTDIR: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY
  WANDB_LOCAL_DIR: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\_wandb_runs


# 2. Preprocesamiento y extracción de features - EJECUTAR

Define las funciones de imagen usadas para construir las características ML.


In [2]:
# ==============================================================================
# 2. FUNCIONES DE PREPROCESAMIENTO DE IMÁGENES
# ==============================================================================

def crop_fundus_rgb(im_pil: Image.Image) -> Image.Image:
    """Recorta la región circular del fondo de ojo."""
    arr = np.array(im_pil)
    g = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    m = (g > 5).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return im_pil
    cnt = max(cnts, key=cv2.contourArea)
    (cx, cy), r = cv2.minEnclosingCircle(cnt)
    cx, cy, r = int(cx), int(cy), int(r * 0.98)
    x0, x1 = max(0, cx - r), min(arr.shape[1], cx + r)
    y0, y1 = max(0, cy - r), min(arr.shape[0], cy + r)
    arr = arr[y0:y1, x0:x1]
    return Image.fromarray(arr)

def clahe_green(im_pil: Image.Image) -> Image.Image:
    """Aplica CLAHE al canal verde (mejora de contraste local)."""
    arr = np.array(im_pil)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    arr[..., 1] = clahe.apply(arr[..., 1])
    return Image.fromarray(arr)

def extract_features_from_image(im_path: str) -> np.ndarray:
    """Extrae características manuales de una imagen."""
    try:
        im = Image.open(im_path).convert("RGB")
        im = crop_fundus_rgb(im)
        im = clahe_green(im)
        arr = np.array(im, dtype=np.float32) / 255.0
        
        # Histogramas RGB (32 bins cada uno)
        hist_r = cv2.calcHist([arr[:,:,0]], [0], None, [32], [0, 1])
        hist_g = cv2.calcHist([arr[:,:,1]], [0], None, [32], [0, 1])
        hist_b = cv2.calcHist([arr[:,:,2]], [0], None, [32], [0, 1])
        
        # Estadísticas de píxeles por canal
        stats_r = [arr[:,:,0].mean(), arr[:,:,0].std(), arr[:,:,0].min(), arr[:,:,0].max()]
        stats_g = [arr[:,:,1].mean(), arr[:,:,1].std(), arr[:,:,1].min(), arr[:,:,1].max()]
        stats_b = [arr[:,:,2].mean(), arr[:,:,2].std(), arr[:,:,2].min(), arr[:,:,2].max()]
        
        # Convertir a escala de grises para LBP
        gray = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        
        # Estadísticas generales
        edge_detection = cv2.Canny(gray, 50, 150)
        edge_ratio = edge_detection.sum() / (gray.size * 255)
        
        # Combinar todas las características
        features = np.concatenate([
            hist_r.flatten(),       # 32 features
            hist_g.flatten(),       # 32 features
            hist_b.flatten(),       # 32 features
            stats_r, stats_g, stats_b,  # 12 features
            [edge_ratio],           # 1 feature
        ])
        
        return features
    except Exception as e:
        print(f"Error extrayendo features de {im_path}: {e}")
        return np.zeros(99)  # 32+32+32+4+4+4+1 = 109 features (ajustar si es necesario)

print("[FUNC] Funciones de preprocesamiento e extracción definidas.")


[FUNC] Funciones de preprocesamiento e extracción definidas.


# 3. Cargar y combinar datasets - EJECUTAR

Construye `df_all` con `path`, `dataset` y `level`.


In [3]:
# ==============================================================================
# 3. CARGAR Y COMBINAR DATASETS
# ==============================================================================

IMAGE_COL_CANDIDATES = {"image","filename","file","img","imagen","Image","File","Filename"}
LABEL_COL_CANDIDATES = {"level","label","labels","grade","diagnosis","target","class","DR","DRGrade","Severity"}

def resolve_img(basename: str, folder: Path):
    """Resuelve la ruta de una imagen."""
    exts = (".png",".jpg",".jpeg",".tif",".tiff",".bmp",".JPG",".JPEG",".PNG")
    s = str(basename)
    if any(s.endswith(e) for e in exts):
        p = folder / s
        return str(p) if p.exists() else None
    for e in exts:
        p = folder / f"{s}{e}"
        if p.exists():
            return str(p)
    return None

def read_train_only(dataset_name: str):
    """Lee el train.csv de un dataset y resuelve rutas de imágenes."""
    root = DATA_ROOT / dataset_name
    csv_path = root / "train.csv"
    if not csv_path.exists():
        print(f"[AVISO] {dataset_name} no tiene train.csv")
        return None
    
    df = pd.read_csv(csv_path)
    
    # Detectar columna de imagen
    img_col = None
    for c in df.columns:
        if c in IMAGE_COL_CANDIDATES or c.lower() in {x.lower() for x in IMAGE_COL_CANDIDATES}:
            img_col = c
            break
    if img_col is None:
        for c in df.columns:
            if re.search(r"(image|file|name|img)", c, re.I):
                img_col = c
                break
    if img_col is None:
        print(f"[AVISO] {csv_path} no tiene columna de imagen")
        return None
    
    # Detectar columna de etiqueta
    y_col = None
    for c in df.columns:
        if c in LABEL_COL_CANDIDATES or c.lower() in {x.lower() for x in LABEL_COL_CANDIDATES}:
            y_col = c
            break
    if y_col is None:
        print(f"[AVISO] {csv_path} no tiene columna de etiqueta")
        return None
    
    df = df[[img_col, y_col]].copy()
    df.rename(columns={img_col:"image", y_col:"level"}, inplace=True)
    df["image"] = df["image"].astype(str)
    
    # Encontrar carpeta de imágenes
    img_dirs = [root/"train", root/"images/train", root]
    img_dir = None
    for d in img_dirs:
        if d.exists():
            img_dir = d
            break
    if img_dir is None:
        cand = [p for p in root.rglob("*") if p.is_dir()]
        best, best_cnt = None, -1
        for p in cand:
            cnt = len([q for q in p.glob("*.*") if q.suffix.lower() in (".png",".jpg",".jpeg",".tif",".tiff",".bmp")])
            if cnt > best_cnt:
                best, best_cnt = p, cnt
        img_dir = best if best is not None else root
    
    df["path"] = df["image"].apply(lambda s: resolve_img(s, img_dir))
    df = df.dropna(subset=["path"]).reset_index(drop=True)
    df["level"] = pd.to_numeric(df["level"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["level"]).astype({"level": int})
    df["dataset"] = dataset_name
    
    print(f"[OK] {dataset_name}/train → filas={len(df)}")
    return df

# Cargar todos los datasets
parts = []
for ds in DATASETS:
    out = read_train_only(ds)
    if out is not None and len(out):
        parts.append(out)

assert parts, "No se encontró ningún train.csv válido."
df_all = pd.concat(parts, ignore_index=True)

# Balanceo opcional
if BALANCE_MAX_PER_CLASS is not None:
    before = df_all["level"].value_counts().sort_index().to_dict()
    df_all = (df_all.groupby("level", group_keys=False)
                     .apply(lambda g: g.sample(min(len(g), BALANCE_MAX_PER_CLASS), random_state=SEED))
                     ).reset_index(drop=True)
    after  = df_all["level"].value_counts().sort_index().to_dict()
    print("\nBalanceo global (tope por clase):")
    print("  Antes :", before)
    print("  Después:", after)

print(f"\nTamaño total combinado: {len(df_all)}")
print(f"Distribución: {df_all['level'].value_counts().sort_index().to_dict()}")


[OK] aptos_2019/train → filas=2930
[OK] eyePACS/train → filas=35126
[OK] idrid/train → filas=413
[OK] messidor/train → filas=1046

Balanceo global (tope por clase):
  Antes : {0: 27846, 1: 2965, 2: 6520, 3: 1172, 4: 1012}
  Después: {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000}

Tamaño total combinado: 5000
Distribución: {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000}


C:\Users\Victoria\AppData\Local\Temp\ipykernel_28300\4020527214.py:99: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), BALANCE_MAX_PER_CLASS), random_state=SEED))


# 4. Features y split train/validación - EJECUTAR

Carga una cache si existe; si no existe, extrae features. También crea `df_va`, `X_va` y `y_va`.


In [4]:
# ==============================================================================
# 4. EXTRACCIÓN DE CARACTERÍSTICAS Y DIVISIÓN TRAIN/VAL
# ==============================================================================

FEATURE_CACHE = OUTDIR / f"features_combined_seed{SEED}_max{BALANCE_MAX_PER_CLASS}.npz"

if FEATURE_CACHE.exists():
    print(f"\n[FEATURES] Cargando cache existente: {FEATURE_CACHE}")
    cache = np.load(FEATURE_CACHE, allow_pickle=True)
    X_all = cache["X_all"]
    y_all = cache["y_all"]
    print(f"  Cache cargada: X_all={X_all.shape}, y_all={y_all.shape}")
else:
    print("\n[FEATURES] Extrayendo características de imágenes...")
    print(f"Total de imágenes: {len(df_all)}")

    features_list = []
    for idx, row in tqdm(df_all.iterrows(), total=len(df_all), desc="Extrayendo features"):
        feat = extract_features_from_image(row["path"])
        features_list.append(feat)

    X_all = np.array(features_list)
    y_all = df_all["level"].values
    np.savez_compressed(FEATURE_CACHE, X_all=X_all, y_all=y_all)
    print(f"  Cache guardada: {FEATURE_CACHE}")

if len(X_all) != len(df_all):
    raise ValueError(f"La cache/features tiene {len(X_all)} filas, pero df_all tiene {len(df_all)}.")

print(f"\nForma de features: {X_all.shape}")
print(f"Rango de valores: [{X_all.min():.4f}, {X_all.max():.4f}]")

y_all = df_all["level"].values

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X_all, y_all))

X_tr = X_all[idx_tr]
y_tr = y_all[idx_tr]
X_va = X_all[idx_va]
y_va = y_all[idx_va]

df_tr = df_all.iloc[idx_tr].reset_index(drop=True)
df_va = df_all.iloc[idx_va].reset_index(drop=True)

print(f"\nSplit interno:")
print(f"  Train: {len(X_tr)} muestras")
print(f"  Val  : {len(X_va)} muestras")
print(f"  Dist. train: {np.bincount(y_tr)}")
print(f"  Dist. val  : {np.bincount(y_va)}")
print(f"  Datasets val: {df_va['dataset'].value_counts().to_dict()}")



[FEATURES] Extrayendo características de imágenes...
Total de imágenes: 5000


Extrayendo features: 100%|██████████| 5000/5000 [37:56<00:00,  2.20it/s]  


  Cache guardada: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\features_combined_seed42_max1000.npz

Forma de features: (5000, 109)
Rango de valores: [0.0000, 10559238.0000]

Split interno:
  Train: 4250 muestras
  Val  : 750 muestras
  Dist. train: [850 850 850 850 850]
  Dist. val  : [150 150 150 150 150]
  Datasets val: {'eyePACS': 607, 'aptos_2019': 91, 'messidor': 30, 'idrid': 22}


# 5. Cargar modelos ML - EJECUTAR

Carga los `.pkl` existentes y solo entrena si falta algún modelo.


In [5]:
# ==============================================================================
# 5. DEFINIR / CARGAR MODELOS ML
# ==============================================================================

# Calcular pesos de clase
class_counts = np.bincount(y_tr)
class_weight = len(y_tr) / (N_CLASSES * class_counts)
class_weight_dict = {i: w for i, w in enumerate(class_weight)}

print("\n[MODELS] Definiendo modelos...")

models_config = {
    "RandomForest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        ),
        "name": "RandomForest"
    },
    "LogisticRegression": {
        "model": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ),
        "name": "LogisticRegression"
    },
    "LinearSVC": {
        "model": LinearSVC(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED,
            dual=False
        ),
        "name": "LinearSVC"
    },
    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=200,
            max_depth=7,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1.0,
            random_state=SEED,
            eval_metric="mlogloss",
            n_jobs=-1
        ),
        "name": "XGBoost"
    }
}

trained_models = {}
for model_key, cfg in models_config.items():
    model_path = OUTDIR / f"{model_key}_model.pkl"

    if model_path.exists():
        print(f"\n  [LOAD] Cargando {cfg['name']} desde {model_path}")
        model = joblib.load(model_path)
        elapsed = 0.0
    else:
        print(f"\n  [TRAIN] Entrenando {cfg['name']} porque no existe {model_path.name}...")
        t0 = time.time()
        model = cfg["model"]
        model.fit(X_tr, y_tr)
        elapsed = time.time() - t0
        joblib.dump(model, model_path)
        print(f"      Modelo guardado: {model_path}")

    trained_models[model_key] = {
        "model": model,
        "name": cfg["name"],
        "train_time": elapsed,
        "model_path": model_path,
    }

print("\n[MODELS] Modelos listos para evaluación/tabla.")



[MODELS] Definiendo modelos...

  [LOAD] Cargando RandomForest desde C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\RandomForest_model.pkl

  [LOAD] Cargando LogisticRegression desde C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\LogisticRegression_model.pkl

  [LOAD] Cargando LinearSVC desde C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\LinearSVC_model.pkl

  [LOAD] Cargando XGBoost desde C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\XGBoost_model.pkl

[MODELS] Modelos listos para evaluación/tabla.


# 6. Tabla de estabilidad por dataset - EJECUTAR

Genera `predicciones_validacion_ml_por_dataset.csv` y `tabla_estabilidad_por_dataset_ml.csv`.


In [6]:
# ==============================================================================
# TABLA DE ESTABILIDAD POR PROCEDENCIA DEL DATO - MODELOS ML
# ==============================================================================
from sklearn.metrics import accuracy_score, f1_score, recall_score, cohen_kappa_score

if "df_va" not in globals():
    raise NameError("Falta df_va. Ejecuta la sección de extracción/split primero.")
if "X_va" not in globals() or "y_va" not in globals():
    raise NameError("Faltan X_va/y_va. Ejecuta la sección de extracción/split primero.")

# Si trained_models no existe, cargar modelos desde OUTDIR.
if "trained_models" not in globals() or not trained_models:
    trained_models = {}
    for model_key in ["RandomForest", "LogisticRegression", "LinearSVC", "XGBoost"]:
        model_path = OUTDIR / f"{model_key}_model.pkl"
        if model_path.exists():
            trained_models[model_key] = {
                "model": joblib.load(model_path),
                "name": model_key,
                "train_time": 0.0,
                "model_path": model_path,
            }
        else:
            print(f"[AVISO] No existe {model_path}; se omitirá {model_key}.")

if not trained_models:
    raise RuntimeError("No hay modelos ML cargados ni archivos *_model.pkl disponibles.")

if len(df_va) != len(X_va):
    raise ValueError(f"df_va tiene {len(df_va)} filas, pero X_va tiene {len(X_va)}.")

pred_rows = []
tabla_rows = []
orden = ["aptos_2019", "eyePACS", "idrid", "messidor"]

for model_key, model_info in trained_models.items():
    model = model_info["model"]
    y_pred = model.predict(X_va)

    df_pred_model = df_va[["path", "dataset", "level"]].copy().reset_index(drop=True)
    df_pred_model["Modelo"] = model_key
    df_pred_model["y_true"] = np.asarray(y_va, dtype=int)
    df_pred_model["y_pred"] = np.asarray(y_pred, dtype=int)
    pred_rows.append(df_pred_model)

    for dataset, g in df_pred_model.groupby("dataset", sort=False):
        y_t = g["y_true"].to_numpy()
        y_p = g["y_pred"].to_numpy()

        recalls = recall_score(
            y_t,
            y_p,
            labels=[0, 1, 2, 3, 4],
            average=None,
            zero_division=0,
        )

        tabla_rows.append({
            "Modelo": model_key,
            "Dataset": dataset,
            "N validación": len(g),
            "Accuracy": accuracy_score(y_t, y_p),
            "F1 macro": f1_score(y_t, y_p, average="macro", zero_division=0),
            "Recall clase 3": recalls[3],
            "Recall clase 4": recalls[4],
            "QWK": cohen_kappa_score(y_t, y_p, weights="quadratic"),
        })

df_pred_val_ml = pd.concat(pred_rows, ignore_index=True)
df_pred_val_ml.to_csv(OUTDIR / "predicciones_validacion_ml_por_dataset.csv", index=False)

tabla_estabilidad_ml = pd.DataFrame(tabla_rows)
cols_metricas = ["Accuracy", "F1 macro", "Recall clase 3", "Recall clase 4", "QWK"]
tabla_estabilidad_ml[cols_metricas] = tabla_estabilidad_ml[cols_metricas].round(3)
tabla_estabilidad_ml["orden_dataset"] = tabla_estabilidad_ml["Dataset"].apply(
    lambda x: orden.index(x) if x in orden else 99
)
tabla_estabilidad_ml["orden_modelo"] = tabla_estabilidad_ml["Modelo"].apply(
    lambda x: ["RandomForest", "LogisticRegression", "LinearSVC", "XGBoost"].index(x)
    if x in ["RandomForest", "LogisticRegression", "LinearSVC", "XGBoost"] else 99
)
tabla_estabilidad_ml = (
    tabla_estabilidad_ml
    .sort_values(["orden_modelo", "orden_dataset"])
    .drop(columns=["orden_modelo", "orden_dataset"])
)

tabla_estabilidad_ml.to_csv(OUTDIR / "tabla_estabilidad_por_dataset_ml.csv", index=False)

print("Archivos generados:")
print("-", OUTDIR / "predicciones_validacion_ml_por_dataset.csv")
print("-", OUTDIR / "tabla_estabilidad_por_dataset_ml.csv")

tabla_estabilidad_ml


Archivos generados:
- C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\predicciones_validacion_ml_por_dataset.csv
- C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\tabla_estabilidad_por_dataset_ml.csv


,Modelo,Dataset,N validación,Accuracy,F1 macro,Recall clase 3,Recall clase 4,QWK
0,RandomForest,aptos_2019,91,0.396,0.483,0.238,0.548,0.351
1,RandomForest,eyePACS,607,0.315,0.312,0.270,0.396,0.232
3,RandomForest,idrid,22,0.318,0.196,0.600,0.400,0.262
2,RandomForest,messidor,30,0.267,0.177,0.333,0.000,0.042
4,LogisticRegression,aptos_2019,91,0.275,0.147,0.095,0.645,-0.049
5,LogisticRegression,eyePACS,607,0.208,0.196,0.243,0.179,0.092
7,LogisticRegression,idrid,22,0.364,0.257,0.400,0.500,0.246
6,LogisticRegression,messidor,30,0.233,0.167,0.556,0.000,0.050
8,LinearSVC,aptos_2019,91,0.275,0.130,0.048,0.710,-0.039
9,LinearSVC,eyePACS,607,0.214,0.198,0.278,0.170,0.057


# 7. Reportes extra, W&B y experimentos individuales - OPCIONAL / PUEDES SALTAR

Estas celdas hacen evaluación global, logging, búsqueda de hiperparámetros o entrenamientos por dataset. No son necesarias para la tabla por procedencia.


## Evaluación global y gráficos - opcional


In [6]:
# ==============================================================================
# 6. EVALUAR MODELOS Y CALCULAR MÉTRICAS (F1, IoU, ROC/AUC)
# ==============================================================================

print("\n[EVAL] Evaluando modelos en validación...")

class_names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
results_summary = []

for model_key, model_info in trained_models.items():
    print(f"\n  [{model_key}]")
    model = model_info["model"]
    
    # Predicciones
    y_pred = model.predict(X_va)
    
    # Probabilidades (si el modelo las soporta)
    try:
        y_prob = model.predict_proba(X_va)
    except Exception:
        # Para modelos que no tienen predict_proba
        # Usamos probabilidades "one-hot" basadas en la predicción
        y_prob = np.zeros((len(X_va), N_CLASSES))
        y_prob[np.arange(len(X_va)), y_pred] = 1.0
    
    # F1 Scores
    f1_macro = f1_score(y_va, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_va, y_pred, average="micro", zero_division=0)
    
    # IoU (Jaccard) por clase
    cm = confusion_matrix(y_va, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None)
    IoU_macro = np.nanmean(IoU)
    
    # ROC/AUC
    y_true_bin = label_binarize(y_va, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
        auc_micro = roc_auc_score(y_true_bin, y_prob, average="micro", multi_class="ovr")
    except Exception as e:
        print(f"[WARN] ROC AUC calculation failed for {model_key}: {e}")
        auc_macro = float('nan')
        auc_micro = float('nan')
    
    # Accuracy
    accuracy = (y_pred == y_va).mean()

    # Quadratic Weighted Kappa (ordinalidad)
    qwk = cohen_kappa_score(y_va, y_pred, weights='quadratic')
    
    print(f"    F1-macro: {f1_macro:.4f} | F1-micro: {f1_micro:.4f}")
    print(f"    IoU-macro: {IoU_macro:.4f}")
    print(f"    AUC-macro: {auc_macro if not np.isnan(auc_macro) else 'nan'} | AUC-micro: {auc_micro if not np.isnan(auc_micro) else 'nan'}")
    print(f"    Accuracy: {accuracy:.4f}")
    print(f"    QWK (Cohen Kappa, weights=quadratic): {qwk:.4f}")
    
    # Guardar reportes
    rep = classification_report(y_va, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = OUTDIR / f"{model_key}_classification_report.csv"
    rep_df.to_csv(rep_path)
    
    iou_df = pd.DataFrame({"clase": np.arange(N_CLASSES), "IoU": IoU})
    iou_path = OUTDIR / f"{model_key}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)
    
    # Gráficos
    # F1 por clase
    fig, ax = plt.subplots(figsize=(6, 4), dpi=140)
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    ax.bar(cls_rows, rep_df.loc[cls_rows, "f1-score"].values if len(cls_rows) else np.zeros(N_CLASSES))
    ax.set_title(f"F1-score por clase ({model_key})")
    ax.set_xlabel("Clase")
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1)
    f1_plot_path = OUTDIR / f"{model_key}_f1_per_class.png"
    plt.tight_layout()
    plt.savefig(f1_plot_path, dpi=140)
    plt.close()
    
    # IoU por clase
    fig, ax = plt.subplots(figsize=(6, 4), dpi=140)
    ax.bar(iou_df["clase"].astype(str).values, iou_df["IoU"].values)
    ax.set_title(f"IoU (Jaccard) por clase ({model_key})")
    ax.set_xlabel("Clase")
    ax.set_ylabel("IoU")
    ax.set_ylim(0, 1)
    iou_plot_path = OUTDIR / f"{model_key}_iou_per_class.png"
    plt.tight_layout()
    plt.savefig(iou_plot_path, dpi=140)
    plt.close()
    
    # ROC Curves
    fig, ax = plt.subplots(figsize=(7, 6), dpi=140)
    try:
        for k in range(N_CLASSES):
            fpr, tpr, _ = roc_curve(y_true_bin[:, k], y_prob[:, k])
            cls_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"Clase {k} (AUC={cls_auc:.3f})")
    except Exception as e:
        print(f"[WARN] No se pudieron dibujar ROC curves para {model_key}: {e}")
    ax.plot([0, 1], [0, 1], "--", lw=1, color="gray")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(f"ROC por clase ({model_key}) · OVR")
    ax.legend(loc="lower right")
    roc_plot_path = OUTDIR / f"{model_key}_roc_curves.png"
    plt.tight_layout()
    plt.savefig(roc_plot_path, dpi=140)
    plt.close()
    
    # Matriz de confusión
    fig, ax = plt.subplots(figsize=(5, 4), dpi=140)
    im = ax.imshow(cm, cmap="viridis")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix ({model_key})")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black", fontsize=8)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    cm_plot_path = OUTDIR / f"{model_key}_confusion_matrix.png"
    plt.tight_layout()
    plt.savefig(cm_plot_path, dpi=140)
    plt.close()
    
    # Almacenar resultados para summary
    results_summary.append({
        "Model": model_key,
        "Accuracy": accuracy,
        "QWK": qwk,
        "F1-Macro": f1_macro,
        "F1-Micro": f1_micro,
        "IoU-Macro": IoU_macro,
        "AUC-Macro": auc_macro,
        "AUC-Micro": auc_micro,
        "Train-Time (s)": model_info["train_time"],
        "y_true": y_va,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "rep_path": rep_path,
        "iou_path": iou_path,
        "f1_plot_path": f1_plot_path,
        "iou_plot_path": iou_plot_path,
        "roc_plot_path": roc_plot_path,
        "cm_plot_path": cm_plot_path
    })

# Resumen comparativo
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["y_true", "y_pred", "y_prob", "rep_path", "iou_path", "f1_plot_path", "iou_plot_path", "roc_plot_path", "cm_plot_path"]} for r in results_summary])
summary_path = OUTDIR / "models_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n[EVAL] Evaluación completada. Resumen guardado.")
print(summary_df)



[EVAL] Evaluando modelos en validación...

  [RandomForest]
    F1-macro: 0.3189 | F1-micro: 0.3227
    IoU-macro: 0.1908
    AUC-macro: 0.6572844444444444 | AUC-micro: 0.6592622222222222
    Accuracy: 0.3227
    QWK (Cohen Kappa, weights=quadratic): 0.2906

  [LogisticRegression]
    F1-macro: 0.2115 | F1-micro: 0.2213
    IoU-macro: 0.1199
    AUC-macro: 0.532468888888889 | AUC-micro: 0.5344951111111111
    Accuracy: 0.2213
    QWK (Cohen Kappa, weights=quadratic): 0.1615

  [LinearSVC]
    F1-macro: 0.2146 | F1-micro: 0.2280
    IoU-macro: 0.1220
    AUC-macro: 0.5174999999999998 | AUC-micro: 0.5175
    Accuracy: 0.2280
    QWK (Cohen Kappa, weights=quadratic): 0.1310

  [XGBoost]
    F1-macro: 0.3348 | F1-micro: 0.3373
    IoU-macro: 0.2024
    AUC-macro: 0.6632666666666667 | AUC-micro: 0.6646853333333334
    Accuracy: 0.3373
    QWK (Cohen Kappa, weights=quadratic): 0.3280

[EVAL] Evaluación completada. Resumen guardado.
                Model  Accuracy       QWK  F1-Macro  F1-Mic

## W&B - opcional


In [7]:
# ==============================================================================
# 7. LOGGING EN WEIGHTS & BIASES (Nuevo proyecto ML)
# ==============================================================================

print("\n[W&B] Iniciando sesión en Weights & Biases...")wandb.login()  # usa la sesi?n local o WANDB_API_KEY

# Crear proyecto separate para ML
ml_project = "retinopatia-diabetica-ml"
ml_entity = "victoria-castro-universidad-peruana-cayetano-heredia"

for result in results_summary:
    model_key = result["Model"]
    print(f"\n  [{model_key}] Subiendo a W&B...")
    
    try:
        # Inicializar run
        run_ml = wandb.init(
            project=ml_project,
            entity=ml_entity,
            name=f"{model_key}_combined",
            tags=["ML", "Combined-Datasets", model_key],
            dir=str(WANDB_LOCAL_DIR),
            config={
                "model": model_key,
                "datasets": DATASETS,
                "n_features": X_tr.shape[1],
                "train_size": len(X_tr),
                "val_size": len(X_va),
                "n_classes": N_CLASSES,
                "balance_max": BALANCE_MAX_PER_CLASS,
                "seed": SEED
            }
        )
        
        # Log de métricas
        run_ml.log({
            "val/accuracy": float(result["Accuracy"]),
            "val/qwk": float(result.get("QWK", float("nan"))),
            "val/f1_macro": float(result["F1-Macro"]),
            "val/f1_micro": float(result["F1-Micro"]),
            "val/iou_macro": float(result["IoU-Macro"]),
            "val/auc_macro": float(result["AUC-Macro"]),
            "val/auc_micro": float(result["AUC-Micro"]),
            "train_time_seconds": float(result["Train-Time (s)"]),
        })
        
        # Log de tablas
        rep_df = pd.read_csv(result["rep_path"])
        iou_df = pd.read_csv(result["iou_path"])
        
        run_ml.log({
            "val/classification_report": wandb.Table(dataframe=rep_df),
            "val/iou_by_class": wandb.Table(dataframe=iou_df),
        })
        
        # Log de gráficos
        run_ml.log({
            "val/f1_plot": wandb.Image(str(result["f1_plot_path"])),
            "val/iou_plot": wandb.Image(str(result["iou_plot_path"])),
            "val/roc_plot": wandb.Image(str(result["roc_plot_path"])),
            "val/confusion_matrix": wandb.Image(str(result["cm_plot_path"]))
        })
        
        # Subir artifacts
        try:
            art_rep = wandb.Artifact(f"classification_report_{model_key}", type="report")
            art_rep.add_file(str(result["rep_path"]))
            run_ml.log_artifact(art_rep)
        except Exception as e:
            print(f"    [W&B] Error subiendo classification_report: {e}")
        
        try:
            art_iou = wandb.Artifact(f"iou_per_class_{model_key}", type="metrics")
            art_iou.add_file(str(result["iou_path"]))
            run_ml.log_artifact(art_iou)
        except Exception as e:
            print(f"    [W&B] Error subiendo iou_per_class: {e}")
        
        # Subir el modelo entrenado como artifact
        try:
            model_path = OUTDIR / f"{model_key}_model.pkl"
            art_model = wandb.Artifact(f"model_{model_key}", type="model")
            art_model.add_file(str(model_path))
            run_ml.log_artifact(art_model)
        except Exception as e:
            print(f"    [W&B] Error subiendo modelo: {e}")
        
        run_ml.finish()
        print(f"    ✓ {model_key} completado en W&B")
    
    except Exception as e:
        print(f"    [ERROR] {model_key}: {e}")
        import traceback
        traceback.print_exc()

print("\n[W&B] Todos los modelos subidos a W&B.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Victoria\_netrc



[W&B] Iniciando sesión en Weights & Biases...


wandb: Currently logged in as: victoria-castro (victoria-castro-universidad-peruana-cayetano-heredia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  [RandomForest] Subiendo a W&B...


train_time_seconds,▁
val/accuracy,▁
val/auc_macro,▁
val/auc_micro,▁
val/f1_macro,▁
val/f1_micro,▁
val/iou_macro,▁
val/qwk,▁
train_time_seconds,0.48102
val/accuracy,0.32267
val/auc_macro,0.65728


...

    ✓ LogisticRegression completado en W&B

  [LinearSVC] Subiendo a W&B...


train_time_seconds,▁
val/accuracy,▁
val/auc_macro,▁
val/auc_micro,▁
val/f1_macro,▁
val/f1_micro,▁
val/iou_macro,▁
val/qwk,▁
train_time_seconds,4.78193
val/accuracy,0.228
val/auc_macro,0.5175


    ✓ LinearSVC completado en W&B

  [XGBoost] Subiendo a W&B...


train_time_seconds,▁
val/accuracy,▁
val/auc_macro,▁
val/auc_micro,▁
val/f1_macro,▁
val/f1_micro,▁
val/iou_macro,▁
val/qwk,▁
train_time_seconds,3.40649
val/accuracy,0.33733
val/auc_macro,0.66327


    ✓ XGBoost completado en W&B

[W&B] Todos los modelos subidos a W&B.


## Experimento APTOS individual - opcional/lento


In [10]:
# ==============================================================================
# EXPERIMENTO ADICIONAL: single dataset (aptos_2019) sin augmentation / sin CLAHE/crop
# - Extrae features desde imágenes originales (sin crop_fundus_rgb ni clahe_green)
# - Entrena RF, LogReg, LinearSVC, XGBoost
# - Evalúa y sube resultados a W&B en proyecto `retinopatia-diabetica-ml`
# ==============================================================================

SINGLE_DATASET = "aptos_2019"  # cambia si quieres otra
EXPERIMENT_OUT = OUTDIR / f"{SINGLE_DATASET}_noaug"
EXPERIMENT_OUT.mkdir(parents=True, exist_ok=True)

print(f"[EXP] Dataset: {SINGLE_DATASET} | Salida: {EXPERIMENT_OUT}")

# 1) Cargar solo ese dataset
df_single = read_train_only(SINGLE_DATASET)
if df_single is None or len(df_single)==0:
    raise RuntimeError(f"No se pudo cargar dataset {SINGLE_DATASET}")
print(f"[EXP] Registros encontrados: {len(df_single)}")

# 2) Definir extractor que NO aplique crop ni CLAHE (datos "originales")
def extract_features_raw(im_path: str) -> np.ndarray:
    try:
        im = Image.open(im_path).convert("RGB")
        arr = np.array(im, dtype=np.float32) / 255.0
        # Histogramas RGB (32 bins)
        hist_r = cv2.calcHist([arr[:,:,0]], [0], None, [32], [0, 1]).flatten()
        hist_g = cv2.calcHist([arr[:,:,1]], [0], None, [32], [0, 1]).flatten()
        hist_b = cv2.calcHist([arr[:,:,2]], [0], None, [32], [0, 1]).flatten()
        # Estadísticas
        stats_r = [arr[:,:,0].mean(), arr[:,:,0].std(), arr[:,:,0].min(), arr[:,:,0].max()]
        stats_g = [arr[:,:,1].mean(), arr[:,:,1].std(), arr[:,:,1].min(), arr[:,:,1].max()]
        stats_b = [arr[:,:,2].mean(), arr[:,:,2].std(), arr[:,:,2].min(), arr[:,:,2].max()]
        gray = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        edge_detection = cv2.Canny(gray, 50, 150)
        edge_ratio = edge_detection.sum() / (gray.size * 255)
        features = np.concatenate([hist_r, hist_g, hist_b, stats_r, stats_g, stats_b, [edge_ratio]])
        return features
    except Exception as e:
        print(f"[EXP] Error extrayendo features de {im_path}: {e}")
        return np.zeros(32*3 + 12 + 1)

# 3) Extraer features (rápido para una sola base, puede tardar según tamaño)
print('[EXP] Extrayendo features (raw) — esto puede tardar unos minutos si hay muchas imágenes')
features = []
paths = df_single['path'].tolist()
for p in tqdm(paths, desc='exp: features'):
    features.append(extract_features_raw(p))
X = np.vstack(features)
y = df_single['level'].values
print('[EXP] Features shape:', X.shape)

# 4) Split estratificado
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X, y))
X_tr, X_va = X[idx_tr], X[idx_va]
y_tr, y_va = y[idx_tr], y[idx_va]
print(f"[EXP] Train/Val: {len(X_tr)}/{len(X_va)}")

# 5) Modelos (mismos hyperparams que antes)
models_config_exp = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                           min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "LinearSVC": LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED, dual=False),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
}

trained_exp = {}
for k, m in models_config_exp.items():
    print(f"[EXP] Entrenando {k}...")
    t0 = time.time()
    m.fit(X_tr, y_tr)
    tt = time.time() - t0
    trained_exp[k] = {'model': m, 'time': tt}
    joblib.dump(m, EXPERIMENT_OUT / f"{k}_model.pkl")
    print(f"[EXP] {k} entrenado en {tt:.1f}s — guardado en {EXPERIMENT_OUT}")

# 6) Evaluar y almacenar reportes + gráficos + subir a W&B
print('[EXP] Evaluando modelos y subiendo resultados a W&B...')

wandb.login()  # usa tu sesión ya configurada
run = wandb.init(project='retinopatia-diabetica-ml', entity='victoria-castro-universidad-peruana-cayetano-heredia',
                 name=f"{SINGLE_DATASET}_noaug_experiment", tags=['ML','no-aug','single-dataset'], dir=str(WANDB_LOCAL_DIR), config={
                     'dataset': SINGLE_DATASET, 'preproc': 'none', 'features': X.shape[1]
                 })

for k, info in trained_exp.items():
    mdl = info['model']
    y_pred = mdl.predict(X_va)
    try:
        y_prob = mdl.predict_proba(X_va)
    except Exception:
        y_prob = np.zeros((len(X_va), N_CLASSES)); y_prob[np.arange(len(X_va)), y_pred] = 1.0
    f1_macro = f1_score(y_va, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_va, y_pred, average='micro', zero_division=0)
    cm = confusion_matrix(y_va, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None); IoU_macro = np.nanmean(IoU)
    y_true_bin = label_binarize(y_va, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr')
    except Exception as e:
        print(f"[EXP] AUC calc failed for {k}: {e}"); auc_macro = float('nan'); auc_micro = float('nan')
    acc = (y_pred == y_va).mean()

    # save report
    rep = classification_report(y_va, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = EXPERIMENT_OUT / f"{k}_classification_report.csv"
    rep_df.to_csv(rep_path)

    iou_df = pd.DataFrame({'clase': np.arange(N_CLASSES), 'IoU': IoU})
    iou_path = EXPERIMENT_OUT / f"{k}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)

    # plots
    f1_plot = EXPERIMENT_OUT / f"{k}_f1_per_class.png"
    plt.figure(figsize=(6,4));
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    plt.bar(cls_rows, rep_df.loc[cls_rows,'f1-score'].values if len(cls_rows) else np.zeros(N_CLASSES)); plt.ylim(0,1); plt.tight_layout(); plt.savefig(f1_plot); plt.close()

    iou_plot = EXPERIMENT_OUT / f"{k}_iou_per_class.png"
    plt.figure(figsize=(6,4)); plt.bar(iou_df['clase'].astype(str).values, iou_df['IoU'].values); plt.ylim(0,1); plt.tight_layout(); plt.savefig(iou_plot); plt.close()

    roc_plot = EXPERIMENT_OUT / f"{k}_roc_curves.png"
    plt.figure(figsize=(7,6));
    try:
        for kk in range(N_CLASSES):
            fpr,tpr,_ = roc_curve(y_true_bin[:,kk], y_prob[:,kk]); plt.plot(fpr,tpr,label=f'Clase {kk}')
    except Exception as e:
        print(f'[EXP] ROC plot failed for {k}: {e}')
    plt.plot([0,1],[0,1],'--',color='gray'); plt.tight_layout(); plt.savefig(roc_plot); plt.close()

    cm_plot = EXPERIMENT_OUT / f"{k}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(5,4), dpi=140)
    im = ax.imshow(cm, cmap='viridis');
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,int(cm[i,j]),ha='center',va='center', color='white' if cm[i,j] > cm.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout(); plt.savefig(cm_plot); plt.close()

    # log to W&B
    run.log({
        'model': k,
        'val/accuracy': float(acc),
        'val/f1_macro': float(f1_macro),
        'val/f1_micro': float(f1_micro),
        'val/iou_macro': float(IoU_macro),
        'val/auc_macro': float(auc_macro) if not np.isnan(auc_macro) else None,
        'val/auc_micro': float(auc_micro) if not np.isnan(auc_micro) else None,
        'train_time_s': float(info['time'])
    })

    run.log({
        f'{k}/classification_report': wandb.Table(dataframe=rep_df.reset_index()),
        f'{k}/iou_by_class': wandb.Table(dataframe=iou_df),
        f'{k}/f1_plot': wandb.Image(str(f1_plot)),
        f'{k}/iou_plot': wandb.Image(str(iou_plot)),
        f'{k}/roc_plot': wandb.Image(str(roc_plot)),
        f'{k}/confusion_matrix': wandb.Image(str(cm_plot)),
    })

    # artifacts
    try:
        art = wandb.Artifact(f"{SINGLE_DATASET}_{k}_report", type='ml-report')
        art.add_file(str(rep_path))
        art.add_file(str(iou_path))
        art.add_file(str(f1_plot))
        art.add_file(str(iou_plot))
        art.add_file(str(roc_plot))
        art.add_file(str(cm_plot))
        run.log_artifact(art)
    except Exception as e:
        print(f"[EXP] Error subiendo artifact para {k}: {e}")

# finish run
run.finish()
print('[EXP] Experimento completado. Revisa W&B y carpeta de salida.')
print('W&B project:', 'retinopatia-diabetica-ml')
print('W&B runs visible en:', f"https://wandb.ai/victoria-castro-universidad-peruana-cayetano-heredia/retinopatia-diabetica-ml")


[EXP] Dataset: aptos_2019 | Salida: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_noaug
[OK] aptos_2019/train → filas=2930
[EXP] Registros encontrados: 2930
[EXP] Extrayendo features (raw) — esto puede tardar unos minutos si hay muchas imágenes


exp: features: 100%|██████████| 2930/2930 [08:24<00:00,  5.81it/s]


[EXP] Features shape: (2930, 109)
[EXP] Train/Val: 2490/440
[EXP] Entrenando RandomForest...
[EXP] RandomForest entrenado en 0.3s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_noaug
[EXP] Entrenando LogisticRegression...
[EXP] LogisticRegression entrenado en 2.1s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_noaug
[EXP] Entrenando LinearSVC...


c:\Users\Victoria\anaconda3\envs\tesis_rd\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[EXP] LinearSVC entrenado en 59.3s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_noaug
[EXP] Entrenando XGBoost...
[EXP] XGBoost entrenado en 2.6s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_noaug
[EXP] Evaluando modelos y subiendo resultados a W&B...


train_time_s,▁▁█▁
val/accuracy,█▁▅█
val/auc_macro,█▃▁█
val/auc_micro,█▁▂█
val/f1_macro,█▁▄█
val/f1_micro,█▁▅█
val/iou_macro,█▁▄█
model,XGBoost
train_time_s,2.57562
val/accuracy,0.75
val/auc_macro,0.87899


[EXP] Experimento completado. Revisa W&B y carpeta de salida.
W&B project: retinopatia-diabetica-ml
W&B runs visible en: https://wandb.ai/victoria-castro-universidad-peruana-cayetano-heredia/retinopatia-diabetica-ml


## Hyperparameter search - opcional/muy lento


In [7]:
# ==============================================================================
# 9. HYPERPARAMETER SEARCH: RandomForest & XGBoost (RandomizedSearchCV)
# ==============================================================================

# Requiere que X_tr, y_tr, X_va, y_va ya estén definidos en el kernel (ejecutados previamente).
try:
    X_tr
    y_tr
    X_va
    y_va
except NameError:
    raise RuntimeError('Variables X_tr/y_tr/X_va/y_va no están definidas. Ejecuta las celdas de extracción y split primero.')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Configuración de búsqueda
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
scoring = 'f1_macro'

# Parámetros para RandomForest
rf_param_dist = {
    'n_estimators': [100, 200, 400],
    'max_depth': [6, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.2, 0.5]
}

# Parámetros para XGBoost
xgb_param_dist = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

n_iter_rf = 24
n_iter_xgb = 30

results_hp = []
HP_OUT = OUTDIR / f"{SINGLE_DATASET}_hp_tuned"
HP_OUT.mkdir(parents=True, exist_ok=True)

# Helper para evaluación y logging local + W&B
import math

def evaluate_and_save(model, name, X_val, y_val, outdir, run_wandb=True):
    y_pred = model.predict(X_val)
    try:
        y_prob = model.predict_proba(X_val)
    except Exception:
        y_prob = np.zeros((len(X_val), N_CLASSES)); y_prob[np.arange(len(X_val)), y_pred] = 1.0
    f1_macro = f1_score(y_val, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_val, y_pred, average='micro', zero_division=0)
    cm = confusion_matrix(y_val, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None); IoU_macro = np.nanmean(IoU)
    y_true_bin = label_binarize(y_val, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr')
    except Exception:
        auc_macro = float('nan'); auc_micro = float('nan')
    acc = (y_pred == y_val).mean()

    # guardar reportes
    rep = classification_report(y_val, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = outdir / f"{name}_classification_report.csv"
    rep_df.to_csv(rep_path)
    iou_df = pd.DataFrame({'clase': np.arange(N_CLASSES), 'IoU': IoU})
    iou_path = outdir / f"{name}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)

    # plots
    f1_plot = outdir / f"{name}_f1_per_class.png"
    plt.figure(figsize=(6,4));
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    plt.bar(cls_rows, rep_df.loc[cls_rows,'f1-score'].values if len(cls_rows) else np.zeros(N_CLASSES)); plt.ylim(0,1); plt.tight_layout(); plt.savefig(f1_plot); plt.close()

    iou_plot = outdir / f"{name}_iou_per_class.png"
    plt.figure(figsize=(6,4)); plt.bar(iou_df['clase'].astype(str).values, iou_df['IoU'].values); plt.ylim(0,1); plt.tight_layout(); plt.savefig(iou_plot); plt.close()

    roc_plot = outdir / f"{name}_roc_curves.png"
    plt.figure(figsize=(7,6));
    try:
        for kk in range(N_CLASSES):
            fpr,tpr,_ = roc_curve(y_true_bin[:,kk], y_prob[:,kk]); plt.plot(fpr,tpr,label=f'Clase {kk}')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'--',color='gray'); plt.tight_layout(); plt.savefig(roc_plot); plt.close()

    cm_plot = outdir / f"{name}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(5,4), dpi=140)
    im = ax.imshow(cm, cmap='viridis')
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,int(cm[i,j]),ha='center',va='center', color='white' if cm[i,j] > cm.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout(); plt.savefig(cm_plot); plt.close()

    # save model
    model_path = outdir / f"{name}_model.pkl"
    joblib.dump(model, model_path)

    # W&B logging
    if run_wandb:
        try:
            run = wandb.init(project='retinopatia-diabetica-ml', entity='victoria-castro-universidad-peruana-cayetano-heredia',
                             name=f"{name}_hp_tuned_{SINGLE_DATASET}", tags=['hp-tuning', name], dir=str(WANDB_LOCAL_DIR), reinit=True,
                             config={'model': name, 'dataset': SINGLE_DATASET})
            run.log({'val/accuracy': float(acc), 'val/f1_macro': float(f1_macro), 'val/f1_micro': float(f1_micro),
                     'val/iou_macro': float(IoU_macro), 'val/auc_macro': float(auc_macro) if not math.isnan(auc_macro) else None,
                     'val/auc_micro': float(auc_micro) if not math.isnan(auc_micro) else None})
            run.log({
                f'{name}/classification_report': wandb.Table(dataframe=rep_df.reset_index()),
                f'{name}/iou_by_class': wandb.Table(dataframe=iou_df),
                f'{name}/f1_plot': wandb.Image(str(f1_plot)),
                f'{name}/iou_plot': wandb.Image(str(iou_plot)),
                f'{name}/roc_plot': wandb.Image(str(roc_plot)),
                f'{name}/confusion_matrix': wandb.Image(str(cm_plot)),
            })
            art = wandb.Artifact(f"{SINGLE_DATASET}_{name}_hp_report", type='hp-report')
            art.add_file(str(rep_path))
            art.add_file(str(iou_path))
            art.add_file(str(f1_plot))
            art.add_file(str(iou_plot))
            art.add_file(str(roc_plot))
            art.add_file(str(cm_plot))
            art.add_file(str(model_path))
            run.log_artifact(art)
            run.finish()
        except Exception as e:
            print(f"[HPO] W&B logging failed for {name}: {e}")

    return dict(name=name, f1_macro=f1_macro, f1_micro=f1_micro, iou_macro=IoU_macro, auc_macro=auc_macro, auc_micro=auc_micro, acc=acc, model_path=str(model_path), rep_path=str(rep_path))


# 1) Randomized search para RandomForest
print('\n[HPO] Buscando RandomForest...')
rf = RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)
rs_rf = RandomizedSearchCV(rf, rf_param_dist, n_iter=n_iter_rf, scoring=scoring, cv=cv, verbose=2, n_jobs=-1, random_state=SEED)
rs_rf.fit(X_tr, y_tr)
print('[HPO] RF best params:', rs_rf.best_params_, 'best score (cv):', rs_rf.best_score_)
best_rf = rs_rf.best_estimator_
res_rf = evaluate_and_save(best_rf, 'RandomForest_hp_tuned', X_va, y_va, HP_OUT)
results_hp.append({'model':'RandomForest','best_params':rs_rf.best_params_,'cv_score':rs_rf.best_score_, **res_rf})

# 2) Randomized search para XGBoost
print('\n[HPO] Buscando XGBoost...')
xgb = XGBClassifier(random_state=SEED, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss')
rs_xgb = RandomizedSearchCV(xgb, xgb_param_dist, n_iter=n_iter_xgb, scoring=scoring, cv=cv, verbose=2, n_jobs=-1, random_state=SEED)
rs_xgb.fit(X_tr, y_tr)
print('[HPO] XGB best params:', rs_xgb.best_params_, 'best score (cv):', rs_xgb.best_score_)
best_xgb = rs_xgb.best_estimator_
res_xgb = evaluate_and_save(best_xgb, 'XGBoost_hp_tuned', X_va, y_va, HP_OUT)
results_hp.append({'model':'XGBoost','best_params':rs_xgb.best_params_,'cv_score':rs_xgb.best_score_, **res_xgb})

# Guardar resultados HPO
hp_df = pd.DataFrame(results_hp)
hp_df.to_csv(HP_OUT / 'hpo_results_summary.csv', index=False)
print('\n[HPO] Búsqueda completada. Resultados guardados en', HP_OUT)
print(hp_df)



[HPO] Buscando RandomForest...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[HPO] RF best params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 10} best score (cv): 0.49178527513480413
[HPO] RF best params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 10} best score (cv): 0.49178527513480413


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


val/accuracy,▁
val/auc_macro,▁
val/auc_micro,▁
val/f1_macro,▁
val/f1_micro,▁
val/iou_macro,▁
val/accuracy,0.72045
val/auc_macro,0.87529
val/auc_micro,0.93099
val/f1_macro,0.54555
val/f1_micro,0.72045



[HPO] Buscando XGBoost...
Fitting 3 folds for each of 30 candidates, totalling 90 fits


c:\Users\Victoria\Desktop\data\.venv\lib\site-packages\xgboost\training.py:199: UserWarning: [22:06:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[HPO] XGB best params: {'subsample': 0.8, 'n_estimators': 400, 'max_depth': 9, 'learning_rate': 0.05, 'colsample_bytree': 0.8} best score (cv): 0.4856389081215191


val/accuracy,▁
val/auc_macro,▁
val/auc_micro,▁
val/f1_macro,▁
val/f1_micro,▁
val/iou_macro,▁
val/accuracy,0.74091
val/auc_macro,0.87849
val/auc_micro,0.938
val/f1_macro,0.52397
val/f1_micro,0.74091



[HPO] Búsqueda completada. Resultados guardados en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\aptos_2019_hp_tuned
          model                                        best_params  cv_score  \
0  RandomForest  {'n_estimators': 100, 'min_samples_split': 10,...  0.491785   
1       XGBoost  {'subsample': 0.8, 'n_estimators': 400, 'max_d...  0.485639   

                    name  f1_macro  f1_micro  iou_macro  auc_macro  auc_micro  \
0  RandomForest_hp_tuned  0.545554  0.720455   0.408326   0.875286   0.930987   
1       XGBoost_hp_tuned  0.523965  0.740909   0.398366   0.878486   0.937998   

        acc                                         model_path  \
0  0.720455  C:\Users\Victoria\Desktop\data\datos_raw\resul...   
1  0.740909  C:\Users\Victoria\Desktop\data\datos_raw\resul...   

                                            rep_path  
0  C:\Users\Victoria\Desktop\data\datos_raw\resul...  
1  C:\Users\Victoria\Desktop\data\datos_raw\resul...  


## Experimento APTOS individual - opcional/lento


In [8]:
# ==============================================================================
# EXPERIMENTO: single dataset (eyePACS) sin augmentation / sin CLAHE/crop
# - Misma lógica usada para aptos_2019, ahora sobre eyePACS
# ==============================================================================

SINGLE_DATASET = "eyePACS"
EXPERIMENT_OUT = OUTDIR / f"{SINGLE_DATASET}_noaug"
EXPERIMENT_OUT.mkdir(parents=True, exist_ok=True)

print(f"[EXP] Dataset: {SINGLE_DATASET} | Salida: {EXPERIMENT_OUT}")

# 1) Cargar solo ese dataset
df_single = read_train_only(SINGLE_DATASET)
if df_single is None or len(df_single)==0:
    raise RuntimeError(f"No se pudo cargar dataset {SINGLE_DATASET}")
print(f"[EXP] Registros encontrados: {len(df_single)}")

# 2) Usar el extractor raw ya definido en sesión (si no existe, reusar el definido anteriormente)
if 'extract_features_raw' not in globals():
    def extract_features_raw(im_path: str) -> np.ndarray:
        try:
            im = Image.open(im_path).convert("RGB")
            arr = np.array(im, dtype=np.float32) / 255.0
            hist_r = cv2.calcHist([arr[:,:,0]], [0], None, [32], [0, 1]).flatten()
            hist_g = cv2.calcHist([arr[:,:,1]], [0], None, [32], [0, 1]).flatten()
            hist_b = cv2.calcHist([arr[:,:,2]], [0], None, [32], [0, 1]).flatten()
            stats_r = [arr[:,:,0].mean(), arr[:,:,0].std(), arr[:,:,0].min(), arr[:,:,0].max()]
            stats_g = [arr[:,:,1].mean(), arr[:,:,1].std(), arr[:,:,1].min(), arr[:,:,1].max()]
            stats_b = [arr[:,:,2].mean(), arr[:,:,2].std(), arr[:,:,2].min(), arr[:,:,2].max()]
            gray = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
            edge_detection = cv2.Canny(gray, 50, 150)
            edge_ratio = edge_detection.sum() / (gray.size * 255)
            features = np.concatenate([hist_r, hist_g, hist_b, stats_r, stats_g, stats_b, [edge_ratio]])
            return features
        except Exception as e:
            print(f"[EXP] Error extrayendo features de {im_path}: {e}")
            return np.zeros(32*3 + 12 + 1)

# 3) Extraer features
print('[EXP] Extrayendo features (raw) — puede tardar algunos minutos según dataset')
features = []
paths = df_single['path'].tolist()
for p in tqdm(paths, desc=f'exp:{SINGLE_DATASET}:features'):
    features.append(extract_features_raw(p))
X = np.vstack(features)
y = df_single['level'].values
print('[EXP] Features shape:', X.shape)

# 4) Split estratificado
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X, y))
X_tr, X_va = X[idx_tr], X[idx_va]
y_tr, y_va = y[idx_tr], y[idx_va]
print(f"[EXP] Train/Val: {len(X_tr)}/{len(X_va)}")

# 5) Entrenar modelos baseline
models_config_exp = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                           min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "LinearSVC": LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED, dual=False),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
}

trained_exp = {}
for k, m in models_config_exp.items():
    print(f"[EXP] Entrenando {k}...")
    t0 = time.time()
    m.fit(X_tr, y_tr)
    tt = time.time() - t0
    trained_exp[k] = {'model': m, 'time': tt}
    joblib.dump(m, EXPERIMENT_OUT / f"{k}_model.pkl")
    print(f"[EXP] {k} entrenado en {tt:.1f}s — guardado en {EXPERIMENT_OUT}")

# 6) Evaluar y subir a W&B (baseline)
print('[EXP] Evaluando modelos y subiendo resultados a W&B...')

wandb.login()  # usa tu sesión ya configurada
run = wandb.init(project='retinopatia-diabetica-ml', entity='victoria-castro-universidad-peruana-cayetano-heredia',
                 name=f"{SINGLE_DATASET}_noaug_experiment", tags=['ML','no-aug','single-dataset'], dir=str(WANDB_LOCAL_DIR), config={
                     'dataset': SINGLE_DATASET, 'preproc': 'none', 'features': X.shape[1]
                 })

results_summary_local = []
for k, info in trained_exp.items():
    mdl = info['model']
    y_pred = mdl.predict(X_va)
    try:
        y_prob = mdl.predict_proba(X_va)
    except Exception:
        y_prob = np.zeros((len(X_va), N_CLASSES)); y_prob[np.arange(len(X_va)), y_pred] = 1.0
    f1_macro = f1_score(y_va, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_va, y_pred, average='micro', zero_division=0)
    cm = confusion_matrix(y_va, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None); IoU_macro = np.nanmean(IoU)
    y_true_bin = label_binarize(y_va, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr')
    except Exception:
        auc_macro = float('nan'); auc_micro = float('nan')
    acc = (y_pred == y_va).mean()

    rep = classification_report(y_va, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = EXPERIMENT_OUT / f"{k}_classification_report.csv"
    rep_df.to_csv(rep_path)
    iou_df = pd.DataFrame({'clase': np.arange(N_CLASSES), 'IoU': IoU})
    iou_path = EXPERIMENT_OUT / f"{k}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)

    # small plots
    f1_plot = EXPERIMENT_OUT / f"{k}_f1_per_class.png"
    plt.figure(figsize=(6,4));
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    plt.bar(cls_rows, rep_df.loc[cls_rows,'f1-score'].values if len(cls_rows) else np.zeros(N_CLASSES)); plt.ylim(0,1); plt.tight_layout(); plt.savefig(f1_plot); plt.close()

    iou_plot = EXPERIMENT_OUT / f"{k}_iou_per_class.png"
    plt.figure(figsize=(6,4)); plt.bar(iou_df['clase'].astype(str).values, iou_df['IoU'].values); plt.ylim(0,1); plt.tight_layout(); plt.savefig(iou_plot); plt.close()

    roc_plot = EXPERIMENT_OUT / f"{k}_roc_curves.png"
    plt.figure(figsize=(7,6));
    try:
        for kk in range(N_CLASSES):
            fpr,tpr,_ = roc_curve(y_true_bin[:,kk], y_prob[:,kk]); plt.plot(fpr,tpr,label=f'Clase {kk}')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'--',color='gray'); plt.tight_layout(); plt.savefig(roc_plot); plt.close()

    cm_plot = EXPERIMENT_OUT / f"{k}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(5,4), dpi=140)
    im = ax.imshow(cm, cmap='viridis');
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,int(cm[i,j]),ha='center',va='center', color='white' if cm[i,j] > cm.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout(); plt.savefig(cm_plot); plt.close()

    run.log({
        'model': k,
        'val/accuracy': float(acc),
        'val/f1_macro': float(f1_macro),
        'val/f1_micro': float(f1_micro),
        'val/iou_macro': float(IoU_macro),
        'val/auc_macro': float(auc_macro) if not np.isnan(auc_macro) else None,
        'val/auc_micro': float(auc_micro) if not np.isnan(auc_micro) else None,
        'train_time_s': float(info['time'])
    })

    run.log({
        f'{k}/classification_report': wandb.Table(dataframe=rep_df.reset_index()),
        f'{k}/iou_by_class': wandb.Table(dataframe=iou_df),
        f'{k}/f1_plot': wandb.Image(str(f1_plot)),
        f'{k}/iou_plot': wandb.Image(str(iou_plot)),
        f'{k}/roc_plot': wandb.Image(str(roc_plot)),
        f'{k}/confusion_matrix': wandb.Image(str(cm_plot)),
    })

    try:
        art = wandb.Artifact(f"{SINGLE_DATASET}_{k}_report", type='ml-report')
        art.add_file(str(rep_path))
        art.add_file(str(iou_path))
        art.add_file(str(f1_plot))
        art.add_file(str(iou_plot))
        art.add_file(str(roc_plot))
        art.add_file(str(cm_plot))
        run.log_artifact(art)
    except Exception as e:
        print(f"[EXP] Error subiendo artifact para {k}: {e}")

    results_summary_local.append({
        'Model': k, 'Accuracy': acc, 'F1-Macro': f1_macro, 'F1-Micro': f1_micro,
        'IoU-Macro': IoU_macro, 'AUC-Macro': auc_macro, 'AUC-Micro': auc_micro,
        'Train-Time (s)': info['time']
    })

run.finish()
print('[EXP] Experimento completado. Revisa W&B y carpeta de salida.')
print('W&B project:', 'retinopatia-diabetica-ml')
print('W&B runs visible en:', f"https://wandb.ai/victoria-castro-universidad-peruana-cayetano-heredia/retinopatia-diabetica-ml")


[EXP] Dataset: eyePACS | Salida: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[OK] eyePACS/train → filas=35126
[EXP] Registros encontrados: 35126
[EXP] Extrayendo features (raw) — puede tardar algunos minutos según dataset
[OK] eyePACS/train → filas=35126
[EXP] Registros encontrados: 35126
[EXP] Extrayendo features (raw) — puede tardar algunos minutos según dataset


exp:eyePACS:features: 100%|██████████| 35126/35126 [3:23:49<00:00,  2.87it/s]  



[EXP] Features shape: (35126, 109)
[EXP] Train/Val: 29857/5269
[EXP] Entrenando RandomForest...
[EXP] RandomForest entrenado en 3.3s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[EXP] Entrenando LogisticRegression...
[EXP] RandomForest entrenado en 3.3s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[EXP] Entrenando LogisticRegression...
[EXP] LogisticRegression entrenado en 9.8s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[EXP] Entrenando LinearSVC...
[EXP] LogisticRegression entrenado en 9.8s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[EXP] Entrenando LinearSVC...
[EXP] LinearSVC entrenado en 54.7s — guardado en C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\eyePACS_noaug
[EXP] Entrenando XGBoost...
[EXP] LinearSVC entrenado en

train_time_s,▁▂█▁
val/accuracy,█▁▇█
val/auc_macro,▇▃▁█
val/auc_micro,█▁▆█
val/f1_macro,█▁▄▃
val/f1_micro,█▁▇█
val/iou_macro,█▁▆▆
model,XGBoost
train_time_s,5.60914
val/accuracy,0.73429
val/auc_macro,0.68042


[EXP] Experimento completado. Revisa W&B y carpeta de salida.
W&B project: retinopatia-diabetica-ml
W&B runs visible en: https://wandb.ai/victoria-castro-universidad-peruana-cayetano-heredia/retinopatia-diabetica-ml


## Experimento IDRiD individual - opcional/lento


In [ ]:
# ==============================================================================
# EXPERIMENTO: single dataset (idrid) sin augmentation / sin CLAHE/crop
# ==============================================================================

SINGLE_DATASET = "idrid"
EXPERIMENT_OUT = OUTDIR / f"{SINGLE_DATASET}_noaug"
EXPERIMENT_OUT.mkdir(parents=True, exist_ok=True)

print(f"[EXP] Dataset: {SINGLE_DATASET} | Salida: {EXPERIMENT_OUT}")

df_single = read_train_only(SINGLE_DATASET)
if df_single is None or len(df_single)==0:
    raise RuntimeError(f"No se pudo cargar dataset {SINGLE_DATASET}")
print(f"[EXP] Registros encontrados: {len(df_single)}")

print('[EXP] Extrayendo features (raw)...')
features = []
paths = df_single['path'].tolist()
for p in tqdm(paths, desc=f'exp:{SINGLE_DATASET}:features'):
    features.append(extract_features_raw(p))
X = np.vstack(features)
y = df_single['level'].values
print('[EXP] Features shape:', X.shape)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X, y))
X_tr, X_va = X[idx_tr], X[idx_va]
y_tr, y_va = y[idx_tr], y[idx_va]
print(f"[EXP] Train/Val: {len(X_tr)}/{len(X_va)}")

models_config_exp = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                           min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "LinearSVC": LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED, dual=False),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
}

trained_exp = {}
for k, m in models_config_exp.items():
    print(f"[EXP] Entrenando {k}...")
    t0 = time.time()
    m.fit(X_tr, y_tr)
    tt = time.time() - t0
    trained_exp[k] = {'model': m, 'time': tt}
    joblib.dump(m, EXPERIMENT_OUT / f"{k}_model.pkl")
    print(f"[EXP] {k} entrenado en {tt:.1f}s")

print('[EXP] Evaluando y subiendo a W&B...')
wandb.login()
run = wandb.init(project='retinopatia-diabetica-ml', entity='victoria-castro-universidad-peruana-cayetano-heredia',
                 name=f"{SINGLE_DATASET}_noaug_experiment", tags=['ML','no-aug','single-dataset'], dir=str(WANDB_LOCAL_DIR), config={
                     'dataset': SINGLE_DATASET, 'preproc': 'none', 'features': X.shape[1]
                 })

for k, info in trained_exp.items():
    mdl = info['model']
    y_pred = mdl.predict(X_va)
    try:
        y_prob = mdl.predict_proba(X_va)
    except Exception:
        y_prob = np.zeros((len(X_va), N_CLASSES)); y_prob[np.arange(len(X_va)), y_pred] = 1.0
    f1_macro = f1_score(y_va, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_va, y_pred, average='micro', zero_division=0)
    cm = confusion_matrix(y_va, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None); IoU_macro = np.nanmean(IoU)
    y_true_bin = label_binarize(y_va, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr')
    except Exception:
        auc_macro = float('nan'); auc_micro = float('nan')
    acc = (y_pred == y_va).mean()

    rep = classification_report(y_va, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = EXPERIMENT_OUT / f"{k}_classification_report.csv"
    rep_df.to_csv(rep_path)
    iou_df = pd.DataFrame({'clase': np.arange(N_CLASSES), 'IoU': IoU})
    iou_path = EXPERIMENT_OUT / f"{k}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)

    f1_plot = EXPERIMENT_OUT / f"{k}_f1_per_class.png"
    plt.figure(figsize=(6,4));
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    plt.bar(cls_rows, rep_df.loc[cls_rows,'f1-score'].values if len(cls_rows) else np.zeros(N_CLASSES)); plt.ylim(0,1); plt.tight_layout(); plt.savefig(f1_plot); plt.close()

    iou_plot = EXPERIMENT_OUT / f"{k}_iou_per_class.png"
    plt.figure(figsize=(6,4)); plt.bar(iou_df['clase'].astype(str).values, iou_df['IoU'].values); plt.ylim(0,1); plt.tight_layout(); plt.savefig(iou_plot); plt.close()

    roc_plot = EXPERIMENT_OUT / f"{k}_roc_curves.png"
    plt.figure(figsize=(7,6));
    try:
        for kk in range(N_CLASSES):
            fpr,tpr,_ = roc_curve(y_true_bin[:,kk], y_prob[:,kk]); plt.plot(fpr,tpr,label=f'Clase {kk}')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'--',color='gray'); plt.tight_layout(); plt.savefig(roc_plot); plt.close()

    cm_plot = EXPERIMENT_OUT / f"{k}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(5,4), dpi=140)
    im = ax.imshow(cm, cmap='viridis');
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,int(cm[i,j]),ha='center',va='center', color='white' if cm[i,j] > cm.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout(); plt.savefig(cm_plot); plt.close()

    run.log({
        'model': k,
        'val/accuracy': float(acc),
        'val/f1_macro': float(f1_macro),
        'val/f1_micro': float(f1_micro),
        'val/iou_macro': float(IoU_macro),
        'val/auc_macro': float(auc_macro) if not np.isnan(auc_macro) else None,
        'val/auc_micro': float(auc_micro) if not np.isnan(auc_micro) else None,
        'train_time_s': float(info['time'])
    })

    run.log({
        f'{k}/classification_report': wandb.Table(dataframe=rep_df.reset_index()),
        f'{k}/iou_by_class': wandb.Table(dataframe=iou_df),
        f'{k}/f1_plot': wandb.Image(str(f1_plot)),
        f'{k}/iou_plot': wandb.Image(str(iou_plot)),
        f'{k}/roc_plot': wandb.Image(str(roc_plot)),
        f'{k}/confusion_matrix': wandb.Image(str(cm_plot)),
    })

    try:
        art = wandb.Artifact(f"{SINGLE_DATASET}_{k}_report", type='ml-report')
        art.add_file(str(rep_path))
        art.add_file(str(iou_path))
        art.add_file(str(f1_plot))
        art.add_file(str(iou_plot))
        art.add_file(str(roc_plot))
        art.add_file(str(cm_plot))
        run.log_artifact(art)
    except Exception as e:
        print(f"[EXP] Error subiendo artifact para {k}: {e}")

run.finish()
print('[EXP] IDRID completado.')


## Experimento Messidor individual - opcional/lento


In [ ]:
# ==============================================================================
# EXPERIMENTO: single dataset (messidor) sin augmentation / sin CLAHE/crop
# ==============================================================================

SINGLE_DATASET = "messidor"
EXPERIMENT_OUT = OUTDIR / f"{SINGLE_DATASET}_noaug"
EXPERIMENT_OUT.mkdir(parents=True, exist_ok=True)

print(f"[EXP] Dataset: {SINGLE_DATASET} | Salida: {EXPERIMENT_OUT}")

df_single = read_train_only(SINGLE_DATASET)
if df_single is None or len(df_single)==0:
    raise RuntimeError(f"No se pudo cargar dataset {SINGLE_DATASET}")
print(f"[EXP] Registros encontrados: {len(df_single)}")

print('[EXP] Extrayendo features (raw)...')
features = []
paths = df_single['path'].tolist()
for p in tqdm(paths, desc=f'exp:{SINGLE_DATASET}:features'):
    features.append(extract_features_raw(p))
X = np.vstack(features)
y = df_single['level'].values
print('[EXP] Features shape:', X.shape)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X, y))
X_tr, X_va = X[idx_tr], X[idx_va]
y_tr, y_va = y[idx_tr], y[idx_va]
print(f"[EXP] Train/Val: {len(X_tr)}/{len(X_va)}")

models_config_exp = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
                                           min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1),
    "LinearSVC": LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED, dual=False),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
}

trained_exp = {}
for k, m in models_config_exp.items():
    print(f"[EXP] Entrenando {k}...")
    t0 = time.time()
    m.fit(X_tr, y_tr)
    tt = time.time() - t0
    trained_exp[k] = {'model': m, 'time': tt}
    joblib.dump(m, EXPERIMENT_OUT / f"{k}_model.pkl")
    print(f"[EXP] {k} entrenado en {tt:.1f}s")

print('[EXP] Evaluando y subiendo a W&B...')
wandb.login()
run = wandb.init(project='retinopatia-diabetica-ml', entity='victoria-castro-universidad-peruana-cayetano-heredia',
                 name=f"{SINGLE_DATASET}_noaug_experiment", tags=['ML','no-aug','single-dataset'], dir=str(WANDB_LOCAL_DIR), config={
                     'dataset': SINGLE_DATASET, 'preproc': 'none', 'features': X.shape[1]
                 })

for k, info in trained_exp.items():
    mdl = info['model']
    y_pred = mdl.predict(X_va)
    try:
        y_prob = mdl.predict_proba(X_va)
    except Exception:
        y_prob = np.zeros((len(X_va), N_CLASSES)); y_prob[np.arange(len(X_va)), y_pred] = 1.0
    f1_macro = f1_score(y_va, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_va, y_pred, average='micro', zero_division=0)
    cm = confusion_matrix(y_va, y_pred, labels=np.arange(N_CLASSES))
    TP = np.diag(cm).astype(float); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP
    IoU = TP / np.clip(TP + FP + FN, 1e-9, None); IoU_macro = np.nanmean(IoU)
    y_true_bin = label_binarize(y_va, classes=np.arange(N_CLASSES))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr')
    except Exception:
        auc_macro = float('nan'); auc_micro = float('nan')
    acc = (y_pred == y_va).mean()

    rep = classification_report(y_va, y_pred, output_dict=True, digits=4, zero_division=0)
    rep_df = pd.DataFrame(rep).T
    rep_path = EXPERIMENT_OUT / f"{k}_classification_report.csv"
    rep_df.to_csv(rep_path)
    iou_df = pd.DataFrame({'clase': np.arange(N_CLASSES), 'IoU': IoU})
    iou_path = EXPERIMENT_OUT / f"{k}_iou_per_class.csv"
    iou_df.to_csv(iou_path, index=False)

    f1_plot = EXPERIMENT_OUT / f"{k}_f1_per_class.png"
    plt.figure(figsize=(6,4));
    cls_rows = [str(i) for i in range(N_CLASSES) if str(i) in rep_df.index]
    plt.bar(cls_rows, rep_df.loc[cls_rows,'f1-score'].values if len(cls_rows) else np.zeros(N_CLASSES)); plt.ylim(0,1); plt.tight_layout(); plt.savefig(f1_plot); plt.close()

    iou_plot = EXPERIMENT_OUT / f"{k}_iou_per_class.png"
    plt.figure(figsize=(6,4)); plt.bar(iou_df['clase'].astype(str).values, iou_df['IoU'].values); plt.ylim(0,1); plt.tight_layout(); plt.savefig(iou_plot); plt.close()

    roc_plot = EXPERIMENT_OUT / f"{k}_roc_curves.png"
    plt.figure(figsize=(7,6));
    try:
        for kk in range(N_CLASSES):
            fpr,tpr,_ = roc_curve(y_true_bin[:,kk], y_prob[:,kk]); plt.plot(fpr,tpr,label=f'Clase {kk}')
    except Exception:
        pass
    plt.plot([0,1],[0,1],'--',color='gray'); plt.tight_layout(); plt.savefig(roc_plot); plt.close()

    cm_plot = EXPERIMENT_OUT / f"{k}_confusion_matrix.png"
    fig, ax = plt.subplots(figsize=(5,4), dpi=140)
    im = ax.imshow(cm, cmap='viridis');
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,int(cm[i,j]),ha='center',va='center', color='white' if cm[i,j] > cm.max()*0.6 else 'black', fontsize=8)
    plt.tight_layout(); plt.savefig(cm_plot); plt.close()

    run.log({
        'model': k,
        'val/accuracy': float(acc),
        'val/f1_macro': float(f1_macro),
        'val/f1_micro': float(f1_micro),
        'val/iou_macro': float(IoU_macro),
        'val/auc_macro': float(auc_macro) if not np.isnan(auc_macro) else None,
        'val/auc_micro': float(auc_micro) if not np.isnan(auc_micro) else None,
        'train_time_s': float(info['time'])
    })

    run.log({
        f'{k}/classification_report': wandb.Table(dataframe=rep_df.reset_index()),
        f'{k}/iou_by_class': wandb.Table(dataframe=iou_df),
        f'{k}/f1_plot': wandb.Image(str(f1_plot)),
        f'{k}/iou_plot': wandb.Image(str(iou_plot)),
        f'{k}/roc_plot': wandb.Image(str(roc_plot)),
        f'{k}/confusion_matrix': wandb.Image(str(cm_plot)),
    })

    try:
        art = wandb.Artifact(f"{SINGLE_DATASET}_{k}_report", type='ml-report')
        art.add_file(str(rep_path))
        art.add_file(str(iou_path))
        art.add_file(str(f1_plot))
        art.add_file(str(iou_plot))
        art.add_file(str(roc_plot))
        art.add_file(str(cm_plot))
        run.log_artifact(art)
    except Exception as e:
        print(f"[EXP] Error subiendo artifact para {k}: {e}")

run.finish()
print('[EXP] MESSIDOR completado.')


In [4]:
# ==============================================================================
# 4. EXTRACCIÓN DE CARACTERÍSTICAS Y DIVISIÓN TRAIN/VAL
# ==============================================================================

print("\n[FEATURES] Extrayendo características de imágenes...")
print(f"Total de imágenes: {len(df_all)}")

# Extraer características (puede tomar varios minutos)
features_list = []
for idx, row in tqdm(df_all.iterrows(), total=len(df_all), desc="Extrayendo features"):
    feat = extract_features_from_image(row["path"])
    features_list.append(feat)

X_all = np.array(features_list)  # (N, F)
y_all = df_all["level"].values   # (N,)

print(f"\nForma de features: {X_all.shape}")
print(f"Rango de valores: [{X_all.min():.4f}, {X_all.max():.4f}]")

# Split estratificado
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
idx_tr, idx_va = next(sss.split(X_all, y_all))

X_tr = X_all[idx_tr]
y_tr = y_all[idx_tr]
X_va = X_all[idx_va]
y_va = y_all[idx_va]

print(f"\nSplit interno:")
print(f"  Train: {len(X_tr)} muestras")
print(f"  Val  : {len(X_va)} muestras")
print(f"  Dist. train: {np.bincount(y_tr)}")
print(f"  Dist. val  : {np.bincount(y_va)}")



[FEATURES] Extrayendo características de imágenes...
Total de imágenes: 5000


Extrayendo features: 100%|██████████| 5000/5000 [48:44<00:00,  1.71it/s]  


Forma de features: (5000, 109)
Rango de valores: [0.0000, 10559238.0000]

Split interno:
  Train: 4250 muestras
  Val  : 750 muestras
  Dist. train: [850 850 850 850 850]
  Dist. val  : [150 150 150 150 150]


## Guardar resumen/configuración - opcional


In [8]:
# ==============================================================================
# 8. GUARDAR CONFIGURACIÓN Y RESUMEN FINAL
# ==============================================================================

print("\n[SAVE] Guardando configuración final...")

# Guardar config.json
cfg = dict(
    mode="machine-learning",
    datasets=DATASETS,
    models=list(trained_models.keys()),
    n_features=X_tr.shape[1],
    train_size=len(X_tr),
    val_size=len(X_va),
    n_classes=N_CLASSES,
    balance_max=BALANCE_MAX_PER_CLASS,
    seed=SEED,
    class_distribution_train=dict(zip(range(N_CLASSES), np.bincount(y_tr).tolist())),
    class_distribution_val=dict(zip(range(N_CLASSES), np.bincount(y_va).tolist())),
)
with open(OUTDIR / "config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print(f"  ✓ config.json guardado")

# Resumen final
print("\n" + "="*80)
print("RESUMEN FINAL DEL ENTRENAMIENTO ML")
print("="*80)
print(f"\nDatasets combinados: {', '.join(DATASETS)}")
print(f"Total de muestras: {len(df_all)}")
print(f"  - Train: {len(X_tr)} ({100*len(X_tr)/len(df_all):.1f}%)")
print(f"  - Val:   {len(X_va)} ({100*len(X_va)/len(df_all):.1f}%)")
print(f"\nCaracterísticas extraídas: {X_tr.shape[1]}")
print(f"Clases: {N_CLASSES}")
print(f"\nModelos entrenados:")
for model_key in trained_models.keys():
    time_taken = trained_models[model_key]["train_time"]
    print(f"  ✓ {model_key} ({time_taken:.1f}s)")

print(f"\nMétricas calculadas:")
print(f"  ✓ F1 Score (Macro, Micro)")
print(f"  ✓ IoU / Jaccard por clase")
print(f"  ✓ ROC/AUC (OVR)")
print(f"  ✓ Confusion Matrix")

print(f"\nOUTDIR: {OUTDIR}")
print(f"  - Modelos: {len(list(OUTDIR.glob('*_model.pkl')))} archivos .pkl")
print(f"  - Reportes: {len(list(OUTDIR.glob('*_classification_report.csv')))} reportes")
print(f"  - Gráficos: {len(list(OUTDIR.glob('*.png')))} imágenes")
print(f"  - Resumen: {summary_path}")

print(f"\nW&B Project: {ml_project}")
print(f"  Accede a: https://wandb.ai/{ml_entity}/{ml_project}")

print("\n" + "="*80)
print("✅ ENTRENAMIENTO ML COMPLETADO")
print("="*80 + "\n")



[SAVE] Guardando configuración final...
  ✓ config.json guardado

RESUMEN FINAL DEL ENTRENAMIENTO ML

Datasets combinados: aptos_2019, eyePACS, idrid, messidor
Total de muestras: 5000
  - Train: 4250 (85.0%)
  - Val:   750 (15.0%)

Características extraídas: 109
Clases: 5

Modelos entrenados:
  ✓ RandomForest (0.5s)
  ✓ LogisticRegression (2.2s)
  ✓ LinearSVC (4.8s)
  ✓ XGBoost (3.4s)

Métricas calculadas:
  ✓ F1 Score (Macro, Micro)
  ✓ IoU / Jaccard por clase
  ✓ ROC/AUC (OVR)
  ✓ Confusion Matrix

OUTDIR: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY
  - Modelos: 4 archivos .pkl
  - Reportes: 4 reportes
  - Gráficos: 16 imágenes
  - Resumen: C:\Users\Victoria\Desktop\data\datos_raw\resultados_ml\COMBINED_TRAIN_ONLY\models_comparison_summary.csv

W&B Project: retinopatia-diabetica-ml
  Accede a: https://wandb.ai/victoria-castro-universidad-peruana-cayetano-heredia/retinopatia-diabetica-ml

✅ ENTRENAMIENTO ML COMPLETADO



In [ ]:
# ==============================================================================
# COMPARACIÓN: Resumen de experimentos y próximos pasos
# ==============================================================================

print("\n" + "="*100)
print("COMPARACIÓN DE EXPERIMENTOS ML - SINGLE DATASET SIN AUGMENTATION")
print("="*100)

# Guardar eyePACS
eyepacs_summary_df = pd.DataFrame(results_summary_local)
eyepacs_summary_df.to_csv(OUTDIR / "eyePACS_noaug" / "models_comparison_summary.csv", index=False)
print("\n✓ Guardado: eyePACS_noaug/models_comparison_summary.csv")

print("\n📊 EYEPACS (35126 imágenes, 29857 train / 5269 val) - Baseline Sin Augmentation")
print(eyepacs_summary_df[['Model','Accuracy','F1-Macro','F1-Micro','IoU-Macro']].to_string(index=False))

print("\n" + "="*100)
print("RESUMEN DE EXPERIMENTOS COMPLETADOS")
print("="*100)
print("""
✅ COMPLETADO:
  1. Experimento APTOS_2019 (baseline sin augmentation)
     - 4 modelos entrenados: RandomForest, LogisticRegression, LinearSVC, XGBoost
     - Mejores métricas: XGBoost (Acc=0.75, F1=0.533, IoU=0.407)
     - W&B run: aptos_2019_noaug_experiment
  
  2. Experimento EYEPACS (baseline sin augmentation)
     - 4 modelos entrenados: RandomForest, LogisticRegression, LinearSVC, XGBoost
     - Mejores métricas: XGBoost (Acc=0.734, F1=0.177, IoU=0.151)
     - W&B run: eyePACS_noaug_experiment
  
  3. HPO en APTOS_2019
     - RandomForest: 24 combinaciones, mejor cv_score=0.492, val F1=0.545
     - XGBoost: 30 combinaciones, mejor cv_score=0.486, val F1=0.523
     - W&B runs: RandomForest_hp_tuned, XGBoost_hp_tuned

📌 OBSERVACIONES:
  • APTOS_2019: F1-Macro ~0.50-0.55 (más balanceado)
  • EYEPACS: F1-Macro ~0.17 (mucho más bajo)
  • Causa probable: eyePACS es >10x más grande, pero tiene imbalance severo de clases
  
📌 PENDIENTE (siguientes datasets):
  1. Experimentos en IDRID (baseline + HPO)
  2. Experimentos en MESSIDOR (baseline + HPO)
  3. Features más ricas: HOG, LBP, GLCM
  4. Cross-validation y ensembles

📂 Carpetas de salida:
  - aptos_2019_noaug/
  - aptos_2019_hp_tuned/
  - eyePACS_noaug/
  Todas bajo: {OUTDIR}
""")
print("="*100)


COMPARACIÓN DE EXPERIMENTOS ML - SINGLE DATASET SIN AUGMENTATION

✓ Guardado: eyePACS_noaug/models_comparison_summary.csv

✓ Encontrada carpeta aptos_2019_noaug
✓ Reconstruido y guardado: aptos_2019_noaug/models_comparison_summary.csv

📊 APTOS_2019 (2930 imágenes, 2490 train / 440 val)
             Model  Accuracy  F1-Macro  F1-Micro  IoU-Macro
         LinearSVC       NaN       NaN       NaN   0.272602
LogisticRegression       NaN       NaN       NaN   0.183085
      RandomForest       NaN       NaN       NaN   0.401696
           XGBoost       NaN       NaN       NaN   0.407234

📊 EYEPACS (35126 imágenes, 29857 train / 5269 val)
             Model  Accuracy  F1-Macro  F1-Micro  IoU-Macro
      RandomForest  0.720440  0.230387  0.720440   0.177967
LogisticRegression  0.245967  0.157351  0.245967   0.091026
         LinearSVC  0.681154  0.188156  0.681154   0.151660
           XGBoost  0.734295  0.177140  0.734295   0.150977

TABLA COMPARATIVA RESUMIDA (Aptos2019 vs EyePACS)
         

C:\Users\Victoria\AppData\Local\Temp\ipykernel_19684\1563760398.py:95: FutureWarning: The behavior of Series.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_aptos = aptos_results.loc[aptos_results['F1-Macro'].idxmax()]


KeyError: nan